# RAG Diagnostic Gym — OpenEnv E2E GRPO Training (Colab)

**OpenEnv `>=0.2.2` · TRL GRPOTrainer · Unsloth QLoRA**

This notebook is designed for judge re-runs and explicitly enforces:
- Online environment-connected reward (no static-label-only training loop)
- OpenEnv-based environment usage
- Baseline vs trained comparison
- Saved evidence plots (`.png`) for reward/loss/baseline-vs-trained
- Composable rubric visibility via `/rubrics`

## What this notebook trains
- **Agent 1 (DiagnosticAgent):** predicts root cause + explanation + confidence
- **Agent 2 (PatchAgent):** proposes config patch + rationale

## Reward design highlights
- Rich signal (not just 0/1): diagnosis quality, patch F1, faithfulness proxy
- Anti-gaming: confidence calibration penalty and distractor penalties
- Composable rubrics via OpenEnv rubric classes and rubric breakdown endpoint

In [1]:
# 1) Install dependencies (OpenEnv latest + training stack)
!pip install -q "openenv-core[core]>=0.2.2"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q fastapi "uvicorn[standard]" websockets pydantic pyyaml datasets matplotlib numpy requests
print("✓ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 38.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.

In [2]:
# 2) Runtime / URL configuration
import os

# If True: use remote deployed environment URL (recommended for submission reruns)
# If False: start local server in this notebook.
USE_REMOTE_ENV = False
REMOTE_ENV_BASE_URL = "https://szyyne-rag-diagnosis-v2.hf.space"

REPO_URL = "https://huggingface.co/spaces/szyyne/RAG_DIagnosis_v2"
REPO_DIR = "/content/rag-diagnostic-gym"

DEMO_MODE = True              # set True for quick smoke run
A1_STEPS = 8 if DEMO_MODE else 250
A2_STEPS = 8 if DEMO_MODE else 250
NUM_BASELINE_RUNS = 8 if DEMO_MODE else 20

BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048
LORA_RANK = 16

print({
    "USE_REMOTE_ENV": USE_REMOTE_ENV,
    "DEMO_MODE": DEMO_MODE,
    "A1_STEPS": A1_STEPS,
    "A2_STEPS": A2_STEPS,
})


{'USE_REMOTE_ENV': False, 'DEMO_MODE': True, 'A1_STEPS': 8, 'A2_STEPS': 8}


In [3]:
# 3) Clone/install repo and prepare artifact directory
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip install -e . --no-deps -q
!mkdir -p plots
print("✓ Repo cloned and installed")

Cloning into '/content/rag-diagnostic-gym'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 78 (delta 26), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (78/78), 84.72 KiB | 1.76 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/rag-diagnostic-gym
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rag-diagnostic-gym (pyproject.toml) ... done
✓ Repo cloned and installed


Wait 30 Secs to let the server run

If the server is not running after 3-4 attempts try restarting the notebook once


In [4]:
# 4) Connect to environment (remote or local) + OpenEnv checks
import subprocess, time, requests

if USE_REMOTE_ENV:
    ENV_HTTP_URL = REMOTE_ENV_BASE_URL.rstrip("/")
    ENV_WS_URL = ENV_HTTP_URL.replace("https://", "wss://").replace("http://", "ws://") + "/ws"
    server = None
else:
    ENV_HTTP_URL = "http://localhost:8000"
    ENV_WS_URL = "ws://localhost:8000/ws"
    server = subprocess.Popen(
        ["uvicorn", "rag_diagnostic_gym.server.environment:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
        cwd=REPO_DIR,
    )
    time.sleep(25)

os.environ["ENV_WS_URL"] = ENV_WS_URL

health = requests.get(f"{ENV_HTTP_URL}/health", timeout=30).json()
tasks = requests.get(f"{ENV_HTTP_URL}/tasks", timeout=30).json()
rubrics = requests.get(f"{ENV_HTTP_URL}/rubrics", timeout=30).json()

print("ENV_HTTP_URL:", ENV_HTTP_URL)
print("ENV_WS_URL  :", ENV_WS_URL)
print("Health      :", health)
print("Tasks       :", list(tasks.keys()))
print("Rubrics keys:", list(rubrics.keys()))

ENV_HTTP_URL: http://localhost:8000
ENV_WS_URL  : ws://localhost:8000/ws
Health      : {'status': 'healthy'}
Tasks       : ['chunking_error_001', 'embedding_mismatch_001', 'hallucination_retrieval_001']
Rubrics keys: ['components', 'difficulty_multipliers', 'normalization']


In [5]:
# 5) Smoke test one full episode via client (client/server separation)
import asyncio
from rag_diagnostic_gym.client import RAGDiagnosticClient
from rag_diagnostic_gym.models import DiagnoseAction, PatchAction

async def smoke_episode():
    async with RAGDiagnosticClient(ENV_WS_URL) as client:
        obs = await client.reset("chunking_error_001")
        obs = await client.step(DiagnoseAction(
            root_cause="chunk_size_too_large",
            explanation="Very large chunks dilute relevant spans in retrieval.",
            confidence=0.9,
        ))
        print("Agent1 step reward:", obs.reward)
        obs = await client.step(PatchAction(
            patch={"chunk_size": 512, "chunk_overlap": 64},
            rationale="Smaller chunks improve retrieval specificity.",
        ))
        print("Agent2 step reward:", obs.reward)
        print("Episode total:", obs.info.get("episode_total_reward"))

await smoke_episode()

Agent1 step reward: 0.2187
Agent2 step reward: 0.39
Episode total: 0.6088


In [6]:
# 6) Random/untrained baseline (required comparator)
import random
from rag_diagnostic_gym.tasks import TASKS

KNOWN_ROOT_CAUSES = list({t["root_cause"] for t in TASKS.values()})

def _obs_info(obs):
    return obs.get("info", {}) if isinstance(obs, dict) else getattr(obs, "info", {})

async def random_episode(task_id: str) -> float:
    async with RAGDiagnosticClient(ENV_WS_URL) as client:
        await client.reset(task_id)
        await client.step(DiagnoseAction(
            root_cause=random.choice(KNOWN_ROOT_CAUSES),
            explanation="Random guess.",
            confidence=0.5,
        ))
        obs = await client.step(PatchAction(patch={}, rationale="No patch."))
        return float(_obs_info(obs).get("episode_total_reward", 0.0))

baseline_rewards = {}
for tid in TASKS:
    vals = [await random_episode(tid) for _ in range(NUM_BASELINE_RUNS)]
    baseline_rewards[tid] = sum(vals) / len(vals)
    print(f"[BASELINE] {tid}: {baseline_rewards[tid]:.4f}")

avg_base = sum(baseline_rewards.values()) / len(baseline_rewards)
print(f"Baseline average: {avg_base:.4f}")

[BASELINE] chunking_error_001: 0.1370
[BASELINE] embedding_mismatch_001: 0.1964
[BASELINE] hallucination_retrieval_001: 0.1095
Baseline average: 0.1476


In [7]:
# 7) Load base model + LoRA (Unsloth)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("✓ Model and LoRA ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✓ Model and LoRA ready


In [8]:
# 8) Train Agent 1 with ONLINE environment rewards
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from training.train_grpo import build_agent1_dataset, make_a1_reward_online
import torch, os

os.makedirs("/content/checkpoints/agent1", exist_ok=True)

ds_a1 = build_agent1_dataset()
if DEMO_MODE:
    ds_a1 = Dataset.from_list(list(ds_a1)[:8])

a1_reward_fn = make_a1_reward_online(ENV_WS_URL)

trainer_a1 = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[a1_reward_fn],
    train_dataset=ds_a1,
    args=GRPOConfig(
        output_dir="/content/checkpoints/agent1",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        max_steps=A1_STEPS,
        learning_rate=1e-5,
        num_generations=4 if DEMO_MODE else 8,
        logging_steps=1,
        save_steps=20,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
    ),
)
print("Training Agent 1 with ONLINE env reward...")
trainer_a1.train()
model.save_pretrained("/content/checkpoints/agent1/final")
print("✓ Agent 1 done")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training Agent 1 with ONLINE env reward...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 1 | Total steps = 8
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarn

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / _fn / mean,rewards / _fn / std
1,-0.000000,0.090225,0.108763,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000009,0.090225,0.108763
2,-0.000000,0.161325,0.038250,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000010,0.161325,0.038250
3,0.000000,0.054675,0.109350,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000018,0.054675,0.109350
4,0.000000,0.090225,0.108763,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000060,0.090225,0.108763
5,-0.000000,0.144900,0.103112,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000053,0.144900,0.103112
6,0.000000,0.035550,0.071100,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000137,0.035550,0.071100
7,0.000000,0.090225,0.108763,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000104,0.090225,0.108763
8,0.000000,0.144900,0.103112,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000154,0.144900,0.103112


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

✓ Agent 1 done


In [9]:
# 9) Train Agent 2 with ONLINE environment rewards
from training.train_grpo import model_fn_from_weights, build_agent2_dataset, make_a2_reward_online
import gc # Import gc for garbage collection

# Build Agent 2 dataset from trained Agent 1 behavior
a1_fn = model_fn_from_weights(model, tokenizer, temperature=0.3)
ds_a2 = build_agent2_dataset(a1_fn)
if DEMO_MODE:
    ds_a2 = Dataset.from_list(list(ds_a2)[:8])

# Clear Agent 1 model from memory before loading Agent 2's
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

# Fresh adapter for Agent 2 stage
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
    device_map="auto" # Add device_map="auto" to handle memory allocation
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

os.makedirs("/content/checkpoints/agent2", exist_ok=True)
a2_reward_fn = make_a2_reward_online(ENV_WS_URL)

trainer_a2 = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[a2_reward_fn],
    train_dataset=ds_a2,
    args=GRPOConfig(
        output_dir="/content/checkpoints/agent2",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        max_steps=A2_STEPS,
        learning_rate=2e-5,
        num_generations=4 if DEMO_MODE else 8,
        logging_steps=1,
        save_steps=20,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
    ),
)
print("Training Agent 2 with ONLINE env reward...")
trainer_a2.train()
model.save_pretrained("/content/checkpoints/agent2/final")
print("✓ Agent 2 done")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 1 | Total steps = 8
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / _fn / mean,rewards / _fn / std
1,0.000000,0.055000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000008,0.055000,0.000000
2,0.000000,0.108600,0.107200,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000007,0.108600,0.107200
3,0.000000,0.055000,0.000000,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000166,0.055000,0.000000
4,0.000001,0.108600,0.107200,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000561,0.108600,0.107200
5,0.000001,0.108600,0.107200,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000920,0.108600,0.107200
6,0.000004,0.162200,0.123784,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.003646,0.162200,0.123784
7,0.000001,0.162200,0.123784,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.000795,0.162200,0.123784
8,0.000004,0.162200,0.123784,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0.003990,0.162200,0.123784


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

✓ Agent 2 done


In [10]:
# 10) Evaluate trained agent vs baseline
from training.train_grpo import evaluate
from rag_diagnostic_gym.tasks import TASKS

# Build a2 inference fn from final stage-2 model
a2_fn = model_fn_from_weights(model, tokenizer)
trained_results = evaluate(a1_fn, a2_fn)

print("\n" + "=" * 72)
print("TRAINED vs RANDOM BASELINE")
print("=" * 72)
print(f"{'Task':<42} {'Baseline':>10} {'Trained':>10} {'Delta':>10}")
print("-" * 72)
for tid in TASKS:
    b = baseline_rewards[tid]
    t = trained_results[tid]["episode_total_reward"]
    d = t - b
    print(f"{tid:<42} {b:>10.4f} {t:>10.4f} {d:>+10.4f}")
print("-" * 72)
avg_trained = sum(v["episode_total_reward"] for v in trained_results.values()) / len(trained_results)
print(f"{'AVERAGE':<42} {avg_base:>10.4f} {avg_trained:>10.4f} {avg_trained - avg_base:>+10.4f}")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



─────────────────────────────────────────────────────────────────
EVALUATION — all 3 tasks   (rewards ∈ [0, 1])
─────────────────────────────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=2


[chunking_error_001]  (easy)
  Root cause  : chunk_size_too_large ✓  (GT: chunk_size_too_large)
  Confidence  : 0.95
  Patch       : {'chunk_size': 512}
  GT patch    : {'chunk_size': 512, 'chunk_overlap': 64}
  Episode reward : 0.4881 / 1.0


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[embedding_mismatch_001]  (medium)
  Root cause  : dimension_mismatch ✗  (GT: embedding_model_mismatch)
  Confidence  : 0.95
  Patch       : {'query_embedding_model': 'text-embedding-ada-002'}
  GT patch    : {'query_encoder': 'text-embedding-ada-002'}
  Episode reward : 0.2320 / 1.0


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[hallucination_retrieval_001]  (hard)
  Root cause  : semantic_drift_score ✗  (GT: query_expansion_semantic_drift_no_reranker)
  Confidence  : 0.95
  Patch       : {'reranker_enabled': True}
  GT patch    : {'query_expansion': False, 'reranker': 'cross-encoder/ms-marco-MiniLM-L-6-v2', 'reranker_top_k': 3}
  Episode reward : 0.2495 / 1.0

─────────────────────────────────────────────────────────────────
  Average episode reward: 0.3232 / 1.0

TRAINED vs RANDOM BASELINE
Task                                         Baseline    Trained      Delta
------------------------------------------------------------------------
chunking_error_001                             0.1370     0.4881    +0.3511
embedding_mismatch_001                         0.1964     0.2320    +0.0356
hallucination_retrieval_001                    0.1095     0.2495    +0.1400
------------------------------------------------------------------------
AVERAGE                                        0.1476     0.3232    +0.1756


In [11]:
# 11) Save evidence plots (.png) and summary numbers
import matplotlib.pyplot as plt
import numpy as np
from train import plot_rewards

OUT_PLOTS = "/content/rag-diagnostic-gym/plots"
os.makedirs(OUT_PLOTS, exist_ok=True)

# 11a) Reward curves from training logs
plot_rewards(
    trainer_a1=trainer_a1,
    trainer_a2=trainer_a2,
    out_path=f"{OUT_PLOTS}/both_agents_reward_curves.png",
)

# 11b) Loss curves (readable axes/units)
def extract_series(trainer, key):
    xs, ys = [], []
    for e in trainer.state.log_history:
        if "step" in e and key in e:
            xs.append(e["step"])
            ys.append(e[key])
    return xs, ys

x1, l1 = extract_series(trainer_a1, "loss")
x2, l2 = extract_series(trainer_a2, "loss")

fig, ax = plt.subplots(figsize=(10, 5))
if l1:
    ax.plot(x1, l1, label="Agent 1 loss", color="#4F46E5")
if l2:
    ax.plot(x2, l2, label="Agent 2 loss", color="#0EA5A4")
ax.set_xlabel("Training step")
ax.set_ylabel("Loss (GRPO objective)")
ax.set_title("Training loss curves")
ax.grid(True, alpha=0.2)
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_PLOTS}/loss_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# 11c) Baseline vs trained on same axes
task_labels = list(TASKS.keys())
base_vals = [baseline_rewards[t] for t in task_labels]
train_vals = [trained_results[t]["episode_total_reward"] for t in task_labels]

x = np.arange(len(task_labels))
w = 0.36
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, base_vals, w, label="Random baseline", color="#B0B3C6")
ax.bar(x + w/2, train_vals, w, label="Trained", color="#4F46E5")
ax.set_xticks(x)
ax.set_xticklabels([f"{t}\n({TASKS[t]['difficulty']})" for t in task_labels], fontsize=9)
ax.set_xlabel("Task")
ax.set_ylabel("Episode reward (0-1)")
ax.set_ylim(0, 1.05)
ax.set_title("Baseline vs trained reward by task")
ax.grid(True, axis="y", alpha=0.2)
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_PLOTS}/baseline_vs_trained.png", dpi=150, bbox_inches="tight")
plt.close(fig)

# 11d) Persist metrics for README/writeup
summary = {
    "mode": "online_env_reward",
    "env_http_url": ENV_HTTP_URL,
    "env_ws_url": ENV_WS_URL,
    "baseline": baseline_rewards,
    "trained": {k: v["episode_total_reward"] for k, v in trained_results.items()},
    "avg_baseline": avg_base,
    "avg_trained": avg_trained,
    "delta": avg_trained - avg_base,
}
import json
with open(f"{OUT_PLOTS}/eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("✓ Saved:")
for name in ["both_agents_reward_curves.png", "loss_curves.png", "baseline_vs_trained.png", "eval_summary.json"]:
    print(" ", f"{OUT_PLOTS}/{name}")

/content/rag-diagnostic-gym/train.py:448: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=9)


✓ Reward curve saved → /content/rag-diagnostic-gym/plots/both_agents_reward_curves.png
✓ Saved:
  /content/rag-diagnostic-gym/plots/both_agents_reward_curves.png
  /content/rag-diagnostic-gym/plots/loss_curves.png
  /content/rag-diagnostic-gym/plots/baseline_vs_trained.png
  /content/rag-diagnostic-gym/plots/eval_summary.json


## 12) Submission checklist notes

- Commit these files from `plots/` to your repo:
  - `both_agents_reward_curves.png`
  - `loss_curves.png`
  - `baseline_vs_trained.png`
  - `eval_summary.json`
- Embed at least the first three PNGs in `README.md` with one-line captions.
- Include baseline/trained numbers from `eval_summary.json` in README/writeup.
- If you use W&B, include the exact run URL in README.

This notebook already enforces **online environment reward** (`make_a1_reward_online` / `make_a2_reward_online`) so the training loop stays connected to the environment.